# 7R-A. Generate the controlled retail environment

This notebook generates the synthetic retail-and-distribution data used for Environment B: the workforce, organisational records, technical activity and separate malicious-activity schedule.

**Saved run:** generator version 1.1, condition `B1_structured`, mode `FINAL`, seed 2105. The configuration and printed output identify this run; the earlier development-run description does not describe this saved copy.

The simulation contains 2,000 employees across 40 stores over 180 days. It includes ordinary activity, legitimate changes in working patterns and the specified malicious scenarios.

**Outputs:** raw activity tables, monthly organisational snapshots, answer files, structural checks, a run manifest and a ZIP package for 7R-B.

No detector is trained and no H1-B decision is made here. CERT r6.2 and the Environment A results are not modified. The saved configuration does not independently establish when the wider evaluation protocol was frozen.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:

from pathlib import Path
import json, hashlib, random, gc, shutil
import numpy as np, pandas as pd

DESIGN_VERSION='1.1'
RUN_MODE='FINAL'              # DEV or FINAL
CONDITION='B1_structured'   # B1_structured / B2_overlap / B3_context_null
FINAL_SEED_INDEX=4
DEV_SEEDS=[1101,1102,1103]; FINAL_SEEDS=[2101,2102,2103,2104,2105]
SEED=DEV_SEEDS[0] if RUN_MODE=='DEV' else FINAL_SEEDS[FINAL_SEED_INDEX]
rng=np.random.default_rng(SEED); random.seed(SEED)

N=2000; DAYS=180; START='2026-01-01'
TRAIN_END=90; VAL_END=120
VAL_MAL=30; TEST_MAL=60; MAL_DAYS=3
ROLES=['Cashier','Warehouse Clerk','Customer Service','Payroll Finance','Store Manager','IT Administrator']
DEPT={'Cashier':'Retail Operations','Warehouse Clerk':'Warehouse','Customer Service':'Customer Service','Payroll Finance':'Finance','Store Manager':'Management','IT Administrator':'Information Technology'}
SHIFT={'Cashier':(8,17),'Warehouse Clerk':(6.5,15.5),'Customer Service':(8.5,17.5),'Payroll Finance':(8,17),'Store Manager':(7.5,18),'IT Administrator':(7,17.5)}
ROLE_W=np.array([.32,.24,.16,.10,.10,.08])

WORK=Path('/kaggle/working'); OUT=WORK/f'retail_env_b_v1_1_{RUN_MODE.lower()}_{CONDITION}_seed{SEED}'
RAW=OUT/'raw'; LDAP=RAW/'LDAP'; ANSW=OUT/'answers'; QA=OUT/'qa'
for p in [OUT,RAW,LDAP,ANSW,QA]: p.mkdir(parents=True,exist_ok=True)
calendar=pd.date_range(START,periods=DAYS,freq='D')
print('Output:',OUT,'| condition:',CONDITION,'| seed:',SEED)


Output: /kaggle/working/retail_env_b_v1_1_final_B1_structured_seed2105 | condition: B1_structured | seed: 2105


## 1. Create the workforce and organisational history

Assign simulated employees to the six roles and store locations, then generate the recorded role transitions and temporary-cover arrangements. Monthly organisational snapshots carry the attributes used by the later feature stage.

These records are generated design inputs, not observations of real employees. The printed counts describe the workforce and changes created in this run.


In [2]:

users=np.array([f'EMP{i:05d}' for i in range(1,N+1)])
roles=rng.choice(ROLES,size=N,p=ROLE_W); stores=rng.integers(1,41,size=N)
wf=pd.DataFrame({'user':users,'role':roles,'store_id':[f'STORE{x:03d}' for x in stores]})
wf['department']=wf.role.map(DEPT); wf['team']=wf.department.str.replace(' ','_')+'_'+wf.store_id; wf['email']=wf.user.str.lower()+'@retail-env-b.example'
managers=wf.loc[wf.role.eq('Store Manager'),'user'].tolist(); bystore=wf.loc[wf.role.eq('Store Manager')].groupby('store_id').user.apply(list).to_dict()
wf['supervisor']=[u if r=='Store Manager' else str(rng.choice(bystore.get(s,managers))) for u,r,s in zip(wf.user,wf.role,wf.store_id)]
wf.to_parquet(OUT/'workforce_master.parquet',index=False)

eligible=wf.loc[~wf.role.eq('Store Manager'),'user'].to_numpy(); ntr=round(.08*N)
tr_users=rng.choice(eligible,ntr,replace=False); tr=[]
for u in tr_users:
    old=wf.loc[wf.user.eq(u),'role'].iloc[0]; new=str(rng.choice([r for r in ROLES if r!=old])); dn=int(rng.integers(105,151))
    tr.append((u,old,new,DEPT[new],dn,calendar[dn-1]))
trans=pd.DataFrame(tr,columns=['user','old_role','new_role','new_department','effective_day_number','effective_date']); trans.to_parquet(OUT/'legitimate_role_transitions.parquet',index=False)
rem=np.setdiff1d(eligible,tr_users); cov_users=rng.choice(rem,round(.06*N),replace=False); cov=[]
for u in cov_users:
    home=wf.loc[wf.user.eq(u),'role'].iloc[0]; cr=str(rng.choice([r for r in ROLES if r!=home])); s=int(rng.integers(100,166)); e=min(s+int(rng.integers(3,11))-1,DAYS); cov.append((u,home,cr,s,e))
cover=pd.DataFrame(cov,columns=['user','home_role','cover_role','start_day_number','end_day_number']); cover.to_parquet(OUT/'legitimate_temporary_cover.parquet',index=False)

def role_at(u,dn,include_cover=True):
    role=wf.loc[wf.user.eq(u),'role'].iloc[0]
    z=trans.loc[trans.user.eq(u)]
    if len(z) and dn>=int(z.iloc[0].effective_day_number): role=z.iloc[0].new_role
    if include_cover:
        c=cover.loc[cover.user.eq(u)]
        if len(c) and int(c.iloc[0].start_day_number)<=dn<=int(c.iloc[0].end_day_number): return c.iloc[0].cover_role,True
    return role,False

for per in pd.period_range(calendar.min(),calendar.max(),freq='M'):
    snap=wf[['user','store_id','email','team','supervisor']].copy(); dn=max(1,(per.start_time-calendar[0]).days+1)
    rr=[role_at(u,dn,False)[0] for u in snap.user]; snap['role']=rr; snap['department']=[DEPT[r] for r in rr]; snap['functional_unit']=snap.department; snap['snapshot']=str(per); snap.to_parquet(LDAP/f'{per}.parquet',index=False)
print('workforce',len(wf),'transitions',len(trans),'temporary cover',len(cover),'LDAP snapshots',len(list(LDAP.glob('*.parquet'))))


workforce 2000 transitions 160 temporary cover 120 LDAP snapshots 6


## 2. Create the malicious-activity schedule

Reserve separate validation and test accounts, with three malicious days per account and no malicious training episodes. The account allocation follows the specified 60% cross-role misuse, 20% globally abnormal and 20% low-and-slow mixture.

The schedule is written to the answer files separately from the raw feature-source channels. It determines the simulated outcome labels; it is not an input feature for the downstream detector.


In [3]:

# ============================================================
# 2. Frozen malicious schedule — exact 60/20/20 account mix
# ============================================================
val_users=rng.choice(users,VAL_MAL,replace=False)
test_users=rng.choice(np.setdiff1d(users,val_users),TEST_MAL,replace=False)

SCENARIO_COUNTS = {
    "validation": {
        "cross_role_misuse": 18,
        "globally_abnormal": 6,
        "low_and_slow": 6,
    },
    "test": {
        "cross_role_misuse": 36,
        "globally_abnormal": 12,
        "low_and_slow": 12,
    },
}

def assign_exact_scenarios(us, counts):
    us=np.asarray(us).copy()
    rng.shuffle(us)
    out={}
    pos=0
    for scen in ["cross_role_misuse","globally_abnormal","low_and_slow"]:
        n=int(counts[scen])
        for u in us[pos:pos+n]:
            out[str(u)]=scen
        pos+=n
    assert pos==len(us)
    return out

m=[]
for part,us,lo,hi in [
    ("validation",val_users,91,120),
    ("test",test_users,121,180),
]:
    assignment=assign_exact_scenarios(us, SCENARIO_COUNTS[part])
    for u in us:
        s=assignment[str(u)]
        ds=np.sort(rng.choice(np.arange(lo,hi+1),MAL_DAYS,replace=False))
        for dn in ds:
            m.append((u,int(dn),calendar[int(dn)-1],part,s))

mal=pd.DataFrame(
    m,columns=["user","day_number","day","partition","scenario"]
)
mal.to_parquet(ANSW/"malicious_user_days.parquet",index=False)
mal.to_csv(ANSW/"malicious_user_days.csv",index=False)

assert mal.query("partition=='validation'").user.nunique()==VAL_MAL
assert mal.query("partition=='test'").user.nunique()==TEST_MAL
assert not (mal.day_number<=TRAIN_END).any()

# Exact account mix checks.
acct_mix=(mal.drop_duplicates(["partition","user"])
          .groupby(["partition","scenario"]).size().unstack(fill_value=0))
for part,counts in SCENARIO_COUNTS.items():
    for scen,n in counts.items():
        assert int(acct_mix.loc[part,scen])==int(n), (part,scen,acct_mix.loc[part,scen],n)

print("Malicious-account scenario mix:")
display(acct_mix)
print("\nMalicious-user-day scenario mix:")
display(mal.groupby(["partition","scenario"]).size().unstack(fill_value=0))


Malicious-account scenario mix:


scenario,cross_role_misuse,globally_abnormal,low_and_slow
partition,,,
test,36,12,12
validation,18,6,6



Malicious-user-day scenario mix:


scenario,cross_role_misuse,globally_abnormal,low_and_slow
partition,,,
test,108,36,36
validation,54,18,18


## 3. Generate active days and behavioural profiles

Build the daily activity plan from the selected condition, role profiles, working patterns and scheduled scenarios. Legitimate exceptions remain benign.

The supplied `B1_structured` run uses the specified role-conditioned behaviour. For cross-role misuse, selected profile parameters blend 20% of the assigned role with 80% of another role, alongside the separate scenario adjustments in the code.

The configuration also defines other conditions, including `B3_context_null`; their presence in the code is not evidence that they were evaluated in this run. The private latent daily plan is retained for generator checks and is not read by 7R-B for feature construction.


In [4]:

P={
'Cashier':dict(logons=2.4,pcs=1.15,device=.08,files=1.8,emails=3,http=18,external=.04,archive=.01,exec=.005),
'Warehouse Clerk':dict(logons=2.1,pcs=1.25,device=.75,files=7,emails=4,http=13,external=.03,archive=.04,exec=.005),
'Customer Service':dict(logons=2.5,pcs=1.2,device=.12,files=2.3,emails=14,http=32,external=.24,archive=.01,exec=.004),
'Payroll Finance':dict(logons=2.3,pcs=1.15,device=.10,files=8,emails=10,http=20,external=.10,archive=.09,exec=.003),
'Store Manager':dict(logons=3,pcs=1.45,device=.18,files=5,emails=18,http=42,external=.18,archive=.04,exec=.005),
'IT Administrator':dict(logons=4.2,pcs=3.3,device=1.35,files=12,emails=12,http=55,external=.12,archive=.15,exec=.09)}
keys=list(P[ROLES[0]]); G={k:np.mean([P[r][k] for r in ROLES]) for k in keys}; alpha={'B1_structured':1.0,'B2_overlap':.35,'B3_context_null':0.0}[CONDITION]
def prof(role): return {k:G[k]+alpha*(P[role][k]-G[k]) for k in keys}
malmap=mal.set_index(['user','day_number']).scenario.to_dict(); rows=[]
for dn,day in enumerate(calendar,1):
    weekend=day.dayofweek>=5
    for u in users:
        role,iscover=role_at(u,dn,True); pa=.18 if weekend else .92
        if weekend and role in {'Cashier','Warehouse Clerk','Store Manager'}: pa+=.16*alpha
        sc=malmap.get((u,dn)); active=True if sc else rng.random()<min(pa,.98)
        if not active: continue
        q=prof(role)
        if CONDITION=='B3_context_null':
            me=(day.day>=26 and rng.random()<.12); st=(day.day in {14,15,28} and rng.random()<.12); nm=(day.dayofweek in {1,3} and rng.random()<.02)
        else:
            me=(role=='Payroll Finance' and day.day>=26 and rng.random()<(0.25+0.50*alpha)); st=(role in {'Warehouse Clerk','Store Manager'} and day.day in {14,15,28} and rng.random()<(0.20+0.35*alpha)); nm=(role=='IT Administrator' and day.dayofweek in {1,3} and rng.random()<(0.03+0.07*alpha))
        if sc=='cross_role_misuse':
            target=str(rng.choice([r for r in ROLES if r!=role])); t=prof(target); q={k:.2*q[k]+.8*t[k] for k in keys}
        gshift=(7.75,17.1); rshift=SHIFT[role]; sh=(gshift[0]+alpha*(rshift[0]-gshift[0]), gshift[1]+alpha*(rshift[1]-gshift[1])); start=float(np.clip(rng.normal(sh[0],.65),0,23)); end=float(np.clip(rng.normal(sh[1],.9),start+.5,23.9))
        if me: end=min(23.9,end+2.2)
        if st: start=max(0,start-1); end=min(23.9,end+1.5)
        if nm: start=float(rng.uniform(0,1)); end=float(rng.uniform(4,6))
        mul=rng.lognormal(0,.25); vals={k:int(max(0,rng.poisson(q[k]*mul))) for k in ['logons','device','files','emails','http']}; pcs=max(1,int(round(rng.normal(q['pcs'],.35))))
        if me: vals['files']+=rng.poisson(10); vals['emails']+=rng.poisson(8)
        if st: vals['files']+=rng.poisson(12); vals['device']+=rng.poisson(1)
        if nm: vals['logons']+=rng.poisson(2); pcs+=int(rng.integers(1,3)); vals['files']+=rng.poisson(8); vals['http']+=rng.poisson(20)
        if sc=='globally_abnormal': vals['logons']+=rng.poisson(5); pcs+=int(rng.integers(2,5)); vals['device']+=rng.poisson(3); vals['files']+=rng.poisson(35); vals['emails']+=rng.poisson(18); vals['http']+=rng.poisson(80); start=min(start,float(rng.uniform(0,5))); end=max(end,float(rng.uniform(21,23.5)))
        elif sc=='cross_role_misuse': vals['files']+=rng.poisson(5); vals['http']+=rng.poisson(10)
        elif sc=='low_and_slow': vals['files']+=rng.poisson(3); vals['emails']+=rng.poisson(2); vals['http']+=rng.poisson(8)
        rows.append((u,dn,day,'train' if dn<=90 else ('validation' if dn<=120 else 'test'),role,iscover,me,st,nm,start,end,max(1,vals['logons']),pcs,vals['device'],vals['files'],vals['emails'],vals['http'],q['external'],q['archive'],q['exec'],sc))
latent=pd.DataFrame(rows,columns=['user','day_number','day','partition','role_for_behaviour','is_temporary_cover','month_end_finance','stocktake','night_maintenance','start_hour','end_hour','logons','pcs','device_n','file_n','email_n','http_n','external_email_p','archive_p','exec_p','scenario_internal_only'])
latent.to_parquet(QA/'latent_user_day_plan_PRIVATE.parquet',index=False); assert latent.query("partition=='train'").scenario_internal_only.isna().all(); print('active user-days',len(latent),'malicious',latent.scenario_internal_only.notna().sum())


active user-days 264893 malicious 270


## 4. Expand the plan into event records

Convert the generated daily plan into logon, device, file, email and HTTP records. HTTP output is written by month to limit memory use.

The downstream feature notebook reconstructs daily features from these raw channels. It does not receive the private plan as a shortcut to the intended features or labels.


In [5]:

def ts(day,h):
    h=float(np.clip(h,0,23.98)); hh=int(h); mm=int((h-hh)*60); return pd.Timestamp(day)+pd.Timedelta(hours=hh,minutes=mm,seconds=int(rng.integers(0,60)))
def times(day,a,b,n):
    if n<=0:return []
    b=max(b,a+.25); return [ts(day,x) for x in np.sort(rng.uniform(a,b,n))]
def pcs(u,n):
    base=int(u.replace('EMP','')); return [f'PC-{(base*7+j*97)%5000:04d}' for j in range(max(1,n))]

L=[];D=[];F=[];E=[]
for r in latent.itertuples(index=False):
    pp=pcs(r.user,r.pcs)
    for j,t in enumerate(times(r.day,r.start_hour,min(r.end_hour,r.start_hour+2),r.logons)):
        pc=pp[j%len(pp)]; L.append((t,r.user,pc,'Logon')); L.append((ts(r.day,max(r.end_hour,float(r.start_hour+.5))),r.user,pc,'Logoff'))
    for j,t in enumerate(times(r.day,r.start_hour,r.end_hour,r.device_n)):
        pc=pp[j%len(pp)]; D.append((t,r.user,pc,'Connect')); D.append((t+pd.Timedelta(minutes=int(rng.integers(5,120))),r.user,pc,'Disconnect'))
    for j,t in enumerate(times(r.day,r.start_hour,r.end_hour,r.file_n)):
        x=rng.random(); ext='.exe' if x<r.exec_p else ('.zip' if x<r.exec_p+r.archive_p else str(rng.choice(['.docx','.pdf','.txt','.xlsx','.csv','.png']))); F.append((t,r.user,pp[j%len(pp)],f'F{rng.integers(1,999999):06d}{ext}','to_removable' if rng.random()<min(.7,.08+r.device_n*.05) else 'local'))
    for j,t in enumerate(times(r.day,r.start_hour,r.end_hour,r.email_n)):
        nr=max(1,int(rng.poisson(1.6))); rec=[]
        for _ in range(nr): rec.append(f'x{rng.integers(1,99999)}@supplier.example' if rng.random()<r.external_email_p else f'emp{rng.integers(1,N+1):05d}@retail-env-b.example')
        E.append((t,r.user,pp[j%len(pp)],r.user.lower()+'@retail-env-b.example',';'.join(rec),int(rng.lognormal(8,.7)),int(rng.binomial(3,min(.5,.07+r.file_n/120)))))
logon=pd.DataFrame(L,columns=['date','user','pc','activity']); device=pd.DataFrame(D,columns=['date','user','pc','activity']); filedf=pd.DataFrame(F,columns=['date','user','pc','filename','direction']); email=pd.DataFrame(E,columns=['date','user','pc','from','to','size','attachments'])
logon.to_parquet(RAW/'logon.parquet',index=False); device.to_parquet(RAW/'device.parquet',index=False); filedf.to_parquet(RAW/'file.parquet',index=False); email.to_parquet(RAW/'email.parquet',index=False)
print('logon',len(logon),'device',len(device),'file',len(filedf),'email',len(email)); del L,D,F,E; gc.collect()


logon 1440896 device 213668 file 1465295 email 2223465


0

In [6]:

# HTTP is written by month to keep memory use bounded.
web=['intranet.retail.example','pos.retail.example','inventory.retail.example','hr.retail.example','finance.retail.example','support.retail.example','news.example','search.example','supplier.example','cloud.example']
latent['month']=latent.day.dt.to_period('M').astype(str); idx=[]
for month,g in latent.groupby('month',sort=True):
    R=[]
    for r in g.itertuples(index=False):
        pp=pcs(r.user,r.pcs)
        for j,t in enumerate(times(r.day,r.start_hour,r.end_hour,r.http_n)):
            dom=str(rng.choice(web))
            if alpha>0:
                if r.role_for_behaviour=='Payroll Finance' and rng.random()<(.35*alpha): dom='finance.retail.example'
                elif r.role_for_behaviour=='Warehouse Clerk' and rng.random()<(.40*alpha): dom='inventory.retail.example'
                elif r.role_for_behaviour=='IT Administrator' and rng.random()<(.40*alpha): dom='support.retail.example'
                elif r.role_for_behaviour=='Cashier' and rng.random()<(.40*alpha): dom='pos.retail.example'
            R.append((t,r.user,pp[j%len(pp)],f'https://{dom}/r/{rng.integers(1,50000)}'))
    h=pd.DataFrame(R,columns=['date','user','pc','url']); fn=f'http_{month}.parquet'; h.to_parquet(RAW/fn,index=False); idx.append((month,len(h),fn)); print(month,len(h)); del R,h; gc.collect()
pd.DataFrame(idx,columns=['month','events','file']).to_csv(QA/'http_partition_index.csv',index=False)


2026-01 1152377
2026-02 1043295
2026-03 1150359
2026-04 1145743
2026-05 1152757
2026-06 1126557


## 5. Check and package the generated world

The structural checks cover the benign training period, separate validation/test malicious accounts, active malicious days, valid users, monthly snapshot count and scenario allocations. The raw channel schemas are checked for forbidden outcome fields.

The notebook then saves the run manifest, file hashes and ZIP package. Passing these checks establishes the tested structural properties, not real-world realism or detection performance.


In [7]:

checks=[]
def ck(n,c,d=''): checks.append({'check':n,'passed':bool(c),'detail':str(d)}); print(('PASS' if c else 'FAIL')+':',n,d)
ck('no malicious training days',not (mal.day_number<=90).any()); ck('validation/test malicious accounts disjoint',set(val_users).isdisjoint(set(test_users))); ck('all malicious days active',set(zip(mal.user,mal.day_number)).issubset(set(zip(latent.user,latent.day_number))))
for n,d in [('logon',logon),('device',device),('file',filedf),('email',email)]:
    ck(n+' contains no label/scenario field',not any(any(x in c.lower() for x in ['label','scenario','malicious','insider']) for c in d.columns)); ck(n+' users valid',set(d.user).issubset(set(users)))
ck('six LDAP snapshots',len(list(LDAP.glob('*.parquet')))==6)
mix_accounts=(mal.drop_duplicates(['partition','user']).groupby(['partition','scenario']).size())
for part,counts in SCENARIO_COUNTS.items():
    for scen,n in counts.items():
        ck(f'exact {part} {scen} account allocation', int(mix_accounts.loc[(part,scen)])==int(n), f"{int(mix_accounts.loc[(part,scen)])}/{n}")
ck('B3 null removes role-specific weekend activation', CONDITION!='B3_context_null' or alpha==0.0)

q=pd.DataFrame(checks); q.to_csv(QA/'generator_structural_checks.csv',index=False); assert q.passed.all()
manifest={'design_version':DESIGN_VERSION,'scenario_mix_policy':'exact_60_20_20_by_account','null_condition_role_signal_alpha':float(alpha),'run_mode':RUN_MODE,'condition':CONDITION,'seed':SEED,'employees':N,'active_user_days':len(latent),'validation_malicious_accounts':len(set(val_users)),'test_malicious_accounts':len(set(test_users)),'malicious_user_days':len(mal),'model_trained':False,'h1_b_verdict':None}; (OUT/'generator_run_manifest.json').write_text(json.dumps(manifest,indent=2))
def sha(p):
    h=hashlib.sha256(); f=open(p,'rb')
    for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    f.close(); return h.hexdigest()
hr=[]
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='output_sha256.csv': hr.append({'relative_path':str(p.relative_to(OUT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(hr).to_csv(OUT/'output_sha256.csv',index=False)
z=shutil.make_archive(str(WORK/OUT.name),'zip',root_dir=OUT)
print(json.dumps(manifest,indent=2)); print('Created',z); print('NO MODEL TRAINED. NO H1-B VERDICT.')


PASS: no malicious training days 
PASS: validation/test malicious accounts disjoint 
PASS: all malicious days active 
PASS: logon contains no label/scenario field 
PASS: logon users valid 
PASS: device contains no label/scenario field 
PASS: device users valid 
PASS: file contains no label/scenario field 
PASS: file users valid 
PASS: email contains no label/scenario field 
PASS: email users valid 
PASS: six LDAP snapshots 
PASS: exact validation cross_role_misuse account allocation 18/18
PASS: exact validation globally_abnormal account allocation 6/6
PASS: exact validation low_and_slow account allocation 6/6
PASS: exact test cross_role_misuse account allocation 36/36
PASS: exact test globally_abnormal account allocation 12/12
PASS: exact test low_and_slow account allocation 12/12
PASS: B3 null removes role-specific weekend activation 
{
  "design_version": "1.1",
  "scenario_mix_policy": "exact_60_20_20_by_account",
  "null_condition_role_signal_alpha": 1.0,
  "run_mode": "FINAL",
  "

## Next stage: 7R-B

Use this world's generated files as the input to 7R-B, which reconstructs the technical and organisational feature blocks. Keep the run manifest and hashes with the raw outputs.

This saved copy represents final seed 2105. Development settings remain in the code for context, but reviewing or editing its notes is not a reason to regenerate the reserved-final worlds. No new generator execution was performed for this documentation edit.
